# define CONTEXT
* CLOSE > base
* exp_returns.sum(lookback= 144) > threshold? bull: (exp_returns.sum(lookback= 144) <  threshold? bear: sideways)

In [1]:
### get data
import pandas as pd
import numpy as np
import talib.abstract as ta

import ccxt

exchange = ccxt.binanceusdm({
	'apiKey': '',
	'secret': '',
})
# turn on dfnet
# hasattr(exchange, 'set_sandbox_mode')
exchange.set_sandbox_mode(True)

df = pd.DataFrame(exchange.fetch_ohlcv(symbol='ETHUSDT', timeframe='4h', limit= 1000))
df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
df.Date= pd.to_datetime(df.Date, unit='ms')
# 定义价格异常的条件，例如价格为负数或超出合理范围
price_columns = ['Open', 'High', 'Low', 'Close']
for col in price_columns:
    df[col] = df[col].mask((df[col] <= 0) | (df[col] > df[col].quantile(0.999)), np.nan)
    df[col] = df[col].ffill()

In [3]:
# calculate the exponential returns
df['exp_return'] = np.log(df['Close'] / df['Close'].shift(1))
df.fillna({'exp_return': 0}, inplace=True)
df.tail(21)

,Date,Open,High,Low,Close,Volume,exp_return
979,2025-10-16 00:00:00,3983.66,4032.37,3974.40,4012.30,1247984.928,0.007161
980,2025-10-16 04:00:00,4012.07,4044.00,3981.36,4000.00,1276731.142,-0.003070
981,2025-10-16 08:00:00,3998.64,4072.97,3944.67,4060.07,1669300.325,0.014906
982,2025-10-16 12:00:00,4060.07,4073.65,3894.16,3905.11,2864118.081,-0.038914
983,2025-10-16 16:00:00,3905.03,3961.40,3851.60,3867.53,2365907.390,-0.009670
984,2025-10-16 20:00:00,3867.43,3893.97,3826.93,3892.02,1048085.184,0.006312
985,2025-10-17 00:00:00,3892.01,3947.28,3887.55,3932.14,666290.006,0.010256
986,2025-10-17 04:00:00,3932.14,3933.59,3754.00,3754.97,1850243.975,-0.046104
987,2025-10-17 08:00:00,3757.27,3809.71,3678.16,3776.07,3234554.311,0.005603
988,2025-10-17 12:00:00,3776.21,3803.94,3715.39,3793.99,2508474.775,0.004734


In [3]:
# define the CONTEXT
'''
df['context'] = np.where(
    df['exp_return'].rolling(window= 126).sum() > 0.05, 'bull',
    np.where(df['exp_return'].rolling(window= 126).sum() < -0.05, 'bear', 'sideways') 
)
'''
# TODO: signal smoothing and dynamic threshold 

"\ndf['context'] = np.where(\n    df['exp_return'].rolling(window= 126).sum() > 0.05, 'bull',\n    np.where(df['exp_return'].rolling(window= 126).sum() < -0.05, 'bear', 'sideways') \n)\n"

In [ ]:
def get_market_regime(series, window= 126, smoothing_period= 5, std_multiplier=0.5):
    """
    Calculates the market regime based on rolling cumulative returns,
    using dynamic thresholds and signal smoothing.

    Args:
        series (pd.Series): The series of returns (preferably log returns).
        window (int): The main look_back window for trend calculation.
        smoothing_period (int): The period for smoothing the momentum score.
        std_multiplier (float): Multiplier for the rolling standard deviation to set thresholds.

    Returns:
        pd.Series: A series with market regime labels ('bull', 'bear', 'sideways').
    """

    # Calculate rolling cumulative returns
    rolling_sum = series.rolling(window=window).sum()

    # Smooth the rolling sum to reduce noise
    # smoothed_momentum = rolling_sum.rolling(window=smoothing_period).mean()
    smoothed_momentum = rolling_sum.ewm(span=smoothing_period, adjust=False).mean() 

    # Calculate dynamic thresholds based on rolling standard deviation
    rolling_std = smoothed_momentum.rolling(window=window).std()
    threshold = std_multiplier * rolling_std

    # Determine market regime based on dynamic thresholds
    conditions = [
        smoothed_momentum > threshold,
        smoothed_momentum < -threshold
    ]
    choices = ['bull', 'bear']
    
    regime = np.select(conditions, choices, default='sideways')

    return pd.Series(regime, index=series.index)    

In [5]:
df['regime'] = get_market_regime(df['exp_return'], window=126, smoothing_period= 8, std_multiplier=0.7)

In [6]:
df.tail(100)

,Date,Open,High,Low,Close,Volume,exp_return,regime
900,2025-10-02 20:00:00,4493.67,4494.50,4456.92,4481.66,7619720.403,-0.002687,sideways
901,2025-10-03 00:00:00,4481.75,4556.81,4461.08,4498.96,3364974.590,0.003853,sideways
902,2025-10-03 04:00:00,4499.02,4506.83,4452.47,4460.72,72592.536,-0.008536,sideways
903,2025-10-03 08:00:00,4460.72,4492.29,4430.00,4476.10,150912.950,0.003442,sideways
904,2025-10-03 12:00:00,4476.10,4530.00,4435.80,4530.00,389532.562,0.011970,sideways
...,...,...,...,...,...,...,...,...
995,2025-10-18 16:00:00,3873.48,3894.66,3852.76,3891.79,650754.637,0.004068,bear
996,2025-10-18 20:00:00,3891.76,3899.40,3868.41,3887.02,343693.104,-0.001226,bear
997,2025-10-19 00:00:00,3886.65,3903.84,3854.71,3897.03,276487.850,0.002572,bear
998,2025-10-19 04:00:00,3897.03,3916.13,3887.01,3906.98,112583.580,0.002550,bear
